# 02 — Train standard CNN

Train the face / not-face CNN, evaluate on a training draw, score the demographic test set, and save `checkpoints/standard_cnn.pt`.

Optional: set `USE_COMET = True` and export `COMET_API_KEY` for experiment tracking.

In [ ]:
from src.paths import setup_project_path
ROOT = setup_project_path()
print("Project root:", ROOT)

In [ ]:
import torch

from config.params import CNN_CHECKPOINT, CNN_PARAMS
from src.device import get_device
from src.data import make_loader
from src.models import make_standard_classifier
from src.train_cnn import train_standard_cnn
from src.evaluate import (
    eval_standard_on_train,
    load_and_plot_test_faces,
    eval_standard_on_demographics,
)

USE_COMET = False

In [ ]:
device = get_device()
loader = make_loader()
standard_classifier = make_standard_classifier(n_outputs=1, device=device)
print(standard_classifier)

In [ ]:
standard_classifier, loss_history = train_standard_cnn(
    standard_classifier, loader, device, params=CNN_PARAMS, use_comet=USE_COMET
)

CNN_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
torch.save(standard_classifier.state_dict(), CNN_CHECKPOINT)
print("Saved", CNN_CHECKPOINT)

In [ ]:
eval_standard_on_train(standard_classifier, loader, device)

In [ ]:
test_faces, keys = load_and_plot_test_faces()
standard_classifier_probs = eval_standard_on_demographics(
    standard_classifier, test_faces, keys, device
)
torch.save(
    {"probs": standard_classifier_probs, "keys": keys},
    CNN_CHECKPOINT.with_name("standard_cnn_demo_probs.pt"),
)
print("Saved demographic probs alongside checkpoint")